In [17]:
import pandas as pd

# Load dataset
df = pd.read_csv("../data/email_evaluation_dataset_sinchana.csv")

# Confirmation message
print("Dataset loaded successfully!")

# Display preview
display(df.head())


Dataset loaded successfully!


,email_text,expected_action,expected_tone
0,Reminder: The project review meeting is schedu...,respond,polite
1,Your electricity bill for August is due by 25t...,respond,urgent
2,Newsletter: Top career tips for engineering st...,ignore,neutral
3,Server CPU usage has crossed 90%. Immediate at...,notify,urgent
4,"Hi, can you please share the updated project d...",respond,polite


In [18]:
df = df.head(100)
df.shape


(100, 3)

In [19]:
df.to_csv("../data/email_evaluation_dataset_sinchana.csv", index=False)


In [20]:
import re

def clean_email_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)   # remove punctuation & numbers
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text


In [21]:
df['clean_text'] = df['email_text'].apply(clean_email_text)


In [22]:
print("Clean_text column created successfully!")
display(df[['email_text', 'clean_text']].head())


Clean_text column created successfully!


,email_text,clean_text
0,Reminder: The project review meeting is schedu...,reminder the project review meeting is schedul...
1,Your electricity bill for August is due by 25t...,your electricity bill for august is due by th ...
2,Newsletter: Top career tips for engineering st...,newsletter top career tips for engineering stu...
3,Server CPU usage has crossed 90%. Immediate at...,server cpu usage has crossed immediate attenti...
4,"Hi, can you please share the updated project d...",hi can you please share the updated project do...


In [2]:
def email_assistant(email_text):
    text = email_text.lower()

    # Urgent condition
    if "urgent" in text or "submit" in text or "deadline" in text:
        return "notify", "urgent"

    # thank you emails
    elif "thank you" in text:
        return "ignore", "polite"

    # Default case
    else:
        return "respond", "neutral"


In [24]:
#Apply email_assistant() to every email
df['assistant_output'] = df['clean_text'].apply(email_assistant)


In [25]:
# Splits the assistant_output tuples into separate action and tone columns for evaluation

df[['predicted_action', 'predicted_tone']] = pd.DataFrame(
    df['assistant_output'].tolist(),
    index=df.index
)


In [26]:
df[['predicted_action', 'predicted_tone']].head()


,predicted_action,predicted_tone
0,respond,neutral
1,respond,neutral
2,respond,neutral
3,respond,neutral
4,respond,neutral


In [27]:

# Displays expected human-labeled action and tone for comparison with assistant predictions
df[['expected_action', 'expected_tone']].head()



,expected_action,expected_tone
0,respond,polite
1,respond,urgent
2,ignore,neutral
3,notify,urgent
4,respond,polite


In [28]:
# Checks whether the assistant's predicted action matches the expected human action and tone matches expected human tone
df['action_correct'] = df['predicted_action'] == df['expected_action']
df['tone_correct'] = df['predicted_tone'] == df['expected_tone']
df.head()

,email_text,expected_action,expected_tone,clean_text,assistant_output,predicted_action,predicted_tone,action_correct,tone_correct
0,Reminder: The project review meeting is schedu...,respond,polite,reminder the project review meeting is schedul...,"(respond, neutral)",respond,neutral,True,False
1,Your electricity bill for August is due by 25t...,respond,urgent,your electricity bill for august is due by th ...,"(respond, neutral)",respond,neutral,True,False
2,Newsletter: Top career tips for engineering st...,ignore,neutral,newsletter top career tips for engineering stu...,"(respond, neutral)",respond,neutral,False,True
3,Server CPU usage has crossed 90%. Immediate at...,notify,urgent,server cpu usage has crossed immediate attenti...,"(respond, neutral)",respond,neutral,False,False
4,"Hi, can you please share the updated project d...",respond,polite,hi can you please share the updated project do...,"(respond, neutral)",respond,neutral,True,False


In [29]:
# Calculates the accuracy of the assistant's action predictions
action_accuracy = df['action_correct'].mean()*100
print("Action Accuracy:", action_accuracy)


Action Accuracy: 42.0


In [30]:
# Calculates the accuracy of the assistant's tone predictions
tone_accuracy = df['tone_correct'].mean()*100
print("Tone Accuracy:", tone_accuracy)


Tone Accuracy: 44.0


In [31]:
# Selects emails where the assistant's action prediction is incorrect
error_df = df[df['action_correct'] == False]
error_df[['clean_text', 'expected_action', 'predicted_action']]
error_df = error_df.reset_index(drop=True)
error_df.head()


,email_text,expected_action,expected_tone,clean_text,assistant_output,predicted_action,predicted_tone,action_correct,tone_correct
0,Newsletter: Top career tips for engineering st...,ignore,neutral,newsletter top career tips for engineering stu...,"(respond, neutral)",respond,neutral,False,True
1,Server CPU usage has crossed 90%. Immediate at...,notify,urgent,server cpu usage has crossed immediate attenti...,"(respond, neutral)",respond,neutral,False,False
2,Your order #4321 has been delivered successfully.,ignore,neutral,your order has been delivered successfully,"(respond, neutral)",respond,neutral,False,True
3,Final reminder: Internship report submission d...,respond,urgent,final reminder internship report submission de...,"(notify, urgent)",notify,urgent,False,True
4,Congratulations! You have won a gift voucher. ...,ignore,neutral,congratulations you have won a gift voucher cl...,"(respond, neutral)",respond,neutral,False,True


In [32]:
# Prints the total number of incorrect action predictions
print("Total incorrect action predictions:", error_df.shape[0])


Total incorrect action predictions: 58


In [33]:
# Save the evaluation results to the data folder
df.to_csv("../data/milestone2_output_sinchana.csv", index=False)


## 1. Which type of emails were hardest to classify?
Ambiguous and indirect emails were hardest to classify.
Example: “Please check this when you have time” – sounds polite but may still require action.

## 2. Why did your rules fail in some cases?
The rules depended only on keywords, so emails without those words were misclassified.
Example: “Server CPU usage has crossed 90%” was urgent but lacked words like urgent or deadline.

## 3. How could an LLM improve this process?
An LLM can understand intent and context, not just keywords.
Example: It can correctly identify “Final reminder: internship report submission” as urgent even without explicit keywords

In [34]:
pip install langsmith


Note: you may need to restart the kernel to use updated packages.


In [35]:
from langsmith import Client
client = Client()

In [36]:
#Define the judge prompt
judge_prompt = """You are an evaluator. Compare the model output with the ideal answer.

check:
1.Action correctness
2.Tone correctness

Give score:
1=correct
0=incorrect
"""

In [37]:
#Run the agent +judge
def evaluate(agent_output,ideal_action,ideal_tone):
    if  agent_output["action"]==ideal_action and agent_output["tone"]==ideal_tone:
        return {"action_score":1,"tone_score":1}
    else:
        return {"action_score":0,"tone_score":0}

In [38]:
agent_output = {"action":"notify",
                "tone":"urgent"}

In [39]:
ideal_action = "notify"
ideal_tone = "urgent"

In [47]:
score = evaluate(agent_output,ideal_action,ideal_tone)
score

{'action_score': 1, 'tone_score': 1}

In [25]:
import pandas as pd

df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,neutral
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,neutral


In [26]:
def email_assistant(email_text):
    text = email_text.lower()

    # Urgent condition
    if "urgent" in text or "submit" in text or "deadline" in text:
        return "notify", "urgent"

    # thank you emails
    elif "thank you" in text or "thanks" in text:
        return "ignore", "polite"

    # Default case
    else:
        return "respond", "neutral"


In [27]:
predictions = [] 
for _, row in df.iterrows(): 
 action, tone = email_assistant(row["body"]) 
predictions.append({ 
"id": row["id"], 
"predicted_intent": action, 
"predicted_tone": tone 
}) 
pred_df = pd.DataFrame(predictions) 
pred_df.head()  

,id,predicted_intent,predicted_tone
0,200,respond,neutral


In [28]:
df[["predicted_intent", "predicted_tone"]] = df["body"].apply(
    lambda x: pd.Series(email_assistant(x))
)

df.head()


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone,predicted_intent,predicted_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,neutral,respond,neutral
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,neutral,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,neutral,respond,neutral


In [29]:
df["intent_correct"] = df["predicted_intent"] == df["ideal_intent"]
df["tone_correct"] = df["predicted_tone"] == df["ideal_tone"]

In [30]:
intent_accuracy = df["intent_correct"].mean() * 100
tone_accuracy = df["tone_correct"].mean() * 100

intent_accuracy, tone_accuracy


(np.float64(92.0), np.float64(92.0))

In [31]:
def evaluate(row):
    score = 0

    if row["predicted_intent"] == row["ideal_intent"]:
        score += 1

    if row["predicted_tone"] == row["ideal_tone"]:
        score += 1

    return score


In [32]:
errors = df[~df["intent_correct"] | ~df["tone_correct"]]
len(errors)
errors.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone,predicted_intent,predicted_tone,intent_correct,tone_correct
8,9,security@bank.com,Account Suspended,"Dear user, we detected a login from a new devi...",high,notify_human,notify,urgent,respond,neutral,False,False
27,28,sales@shop.com,Event Invitation,"Dear user, we detected a login from a new devi...",low,notify_human,notify,urgent,respond,neutral,False,False
40,41,news@techblog.com,Survey,"Dear user, we detected a login from a new devi...",low,notify_human,notify,urgent,respond,neutral,False,False
41,42,orders@ecom.com,Payment Overdue,"Dear user, we detected a login from a new devi...",medium,notify_human,notify,urgent,respond,neutral,False,False
46,47,support@cloud.com,Monthly Report,"Dear user, we detected a login from a new devi...",low,notify_human,notify,urgent,respond,neutral,False,False


In [33]:
df.to_csv(
    "../data/milestone2_output_sinchana.csv",
    index=False
)
